# step3_feature_mlp

**Owner:** P4  
**Purpose:** Feature-based MLP (history, category, entities, article age), with feature ablation.  
**Inputs:** `data/processed/` from steps 1 and 3  
**Outputs:** `outputs/results_step3_feature_mlp.csv`

> Runtime: Colab. Run all cells top to bottom; keep outputs saved.

In [4]:
# ==========================================================
# 1. GOOGLE COLAB SETUP
# ==========================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import subprocess
from pathlib import Path

# Clone the GitHub project into Colab
PROJECT_ROOT = Path("/content/BT4222-Project-Group-4--MIND")

if not (PROJECT_ROOT / "config.py").exists():
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git",
        str(PROJECT_ROOT)
    ], check=True)

sys.path.insert(0, str(PROJECT_ROOT))

# Shared Google Drive project folder
DATA_ROOT = Path("/content/drive/MyDrive/BT4222 G4/BT4222_Project")

os.environ["MIND_PROJECT_DIR"] = str(DATA_ROOT)

import config
import importlib
importlib.reload(config)

PROCESSED_DIR = Path(config.PROCESSED_DIR)

print("Repository found:", (PROJECT_ROOT / "config.py").exists())
print("Project folder found:", DATA_ROOT.is_dir())
print("Processed folder found:", PROCESSED_DIR.is_dir())

os.environ["MIND_PROJECT_DIR"] = str(DATA_ROOT)

import config
import importlib
importlib.reload(config)

PROCESSED_DIR = Path(config.PROCESSED_DIR)

print("Repository found:", (PROJECT_ROOT / "config.py").exists())
print("Processed folder found:", PROCESSED_DIR.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repository found: True
Project folder found: True
Processed folder found: True
Repository found: True
Processed folder found: True


In [5]:
# ==========================================================
# 2. NEW FEATURE MLP ARCHITECTURE
# ==========================================================

import torch
import torch.nn as nn

class FeatureMLP(nn.Module):
    def __init__(self, n_categories, n_subcategories):
        super().__init__()

        # Shared embeddings for users' history and candidate news
        self.category_embedding = nn.Embedding(
            n_categories, 8, padding_idx=0
        )
        self.subcategory_embedding = nn.Embedding(
            n_subcategories, 16, padding_idx=0
        )

        # 5 numerical + 24 user + 24 article = 53 inputs
        self.mlp = nn.Sequential(
            nn.Linear(53, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def average_history(self, history_ids, embedding):
        # Ignore padding and unknown category IDs (0)
        mask = (history_ids != 0).unsqueeze(-1)
        vectors = embedding(history_ids) * mask

        counts = mask.sum(dim=1).clamp(min=1)
        return vectors.sum(dim=1) / counts

    def forward(
        self,
        category_idx,
        subcategory_idx,
        history_category_idx,
        history_subcategory_idx,
        numerical_features
    ):
        # Article embeddings: 8 + 16 dimensions
        article_cat = self.category_embedding(category_idx)
        article_subcat = self.subcategory_embedding(
            subcategory_idx
        )

        # User history embeddings: 8 + 16 dimensions
        user_cat = self.average_history(
            history_category_idx,
            self.category_embedding
        )
        user_subcat = self.average_history(
            history_subcategory_idx,
            self.subcategory_embedding
        )

        # Combine all 53 dimensions
        x = torch.cat([
            numerical_features,
            user_cat,
            user_subcat,
            article_cat,
            article_subcat
        ], dim=1)

        return self.mlp(x).squeeze(-1)


print("Feature MLP architecture defined successfully!")

Feature MLP architecture defined successfully!


In [6]:

# ==========================================================
# 3. TEST FEATURE MLP WITH DUMMY DATA
# ==========================================================

import torch

torch.manual_seed(42)

# Example vocabulary sizes (only for testing)
model = FeatureMLP(
    n_categories=20,
    n_subcategories=300
)

model.eval()

# Test 5 candidate articles
category_idx = torch.tensor([1, 2, 3, 1, 2])
subcategory_idx = torch.tensor([10, 20, 30, 10, 20])

# Previously clicked article categories
# Zeros represent padding / unknown categories
history_category_idx = torch.tensor([
    [1, 2, 3, 0],
    [2, 2, 0, 0],
    [3, 1, 2, 1],
    [0, 0, 0, 0],
    [1, 1, 2, 0]
])

history_subcategory_idx = torch.tensor([
    [10, 20, 30, 0],
    [20, 20, 0, 0],
    [30, 10, 20, 10],
    [0, 0, 0, 0],
    [10, 10, 20, 0]
])

# Five numerical features per article
numerical_features = torch.rand(5, 5)

with torch.no_grad():
    scores = model(
        category_idx,
        subcategory_idx,
        history_category_idx,
        history_subcategory_idx,
        numerical_features
    )

print("Output shape:", scores.shape)
print("Scores:", scores)
print("All scores finite:", torch.isfinite(scores).all().item())

assert scores.shape == (5,)
assert torch.isfinite(scores).all()

print("\nMLP architecture test PASSED!")


Output shape: torch.Size([5])
Scores: tensor([0.2266, 0.2672, 0.2051, 0.2098, 0.2728])
All scores finite: True

MLP architecture test PASSED!


In [7]:

# ==========================================================
# 4. FIVE-WAY SOFTMAX TRAINING LOSS
# ==========================================================

import torch
import torch.nn.functional as F

def five_way_loss(scores, clicked_positions):
    """
    scores: [batch_size, 5]
    clicked_positions: [batch_size]

    Each group contains 1 clicked article
    and 4 non-clicked articles.
    """
    assert scores.ndim == 2
    assert scores.shape[1] == 5

    return F.cross_entropy(scores, clicked_positions)


# Test using the dummy model from Step 3
model.train()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

# All five articles belong to one example impression,
# so they must share the same user history.
group_hist_cat = history_category_idx[:1].repeat(5, 1)
group_hist_subcat = history_subcategory_idx[:1].repeat(5, 1)

# Generate five scores with gradients enabled
scores = model(
    category_idx,
    subcategory_idx,
    group_hist_cat,
    group_hist_subcat,
    numerical_features
).unsqueeze(0)

# For this example, article 0 was clicked
clicked_positions = torch.tensor([0])

loss = five_way_loss(scores, clicked_positions)

# Test one optimisation step
optimizer.zero_grad()
loss.backward()
optimizer.step()

assert torch.isfinite(loss).item()

print("Scores shape:", scores.shape)
print("Training loss:", round(loss.item(), 4))
print("Five-way softmax training test PASSED!")


Scores shape: torch.Size([1, 5])
Training loss: 1.5824
Five-way softmax training test PASSED!


In [8]:

# ==========================================================
# 5. BUILD FIVE-WAY TRAINING GROUPS
# ==========================================================

import numpy as np
import pandas as pd

def build_five_way_groups(df, seed=42):
    """
    Build one-click + four-non-click training groups.

    All five articles must belong to the same impression.
    Impressions with multiple clicks generate multiple groups.
    """
    rng = np.random.default_rng(seed)
    df = df.reset_index(drop=True)

    groups = []
    targets = []
    skipped = 0

    for _, slate in df.groupby("impression_id", sort=False):

        positives = slate[slate["label"] == 1].index.to_numpy()

        negatives = (
            slate[slate["label"] == 0]
            .drop_duplicates(subset="news_id")
        )

        for positive_idx in positives:

            positive_news = df.at[positive_idx, "news_id"]

            available = negatives[
                negatives["news_id"] != positive_news
            ].index.to_numpy()

            if len(available) < 4:
                skipped += 1
                continue

            sampled_negatives = rng.choice(
                available, size=4, replace=False
            )

            five_indices = np.concatenate([
                [positive_idx], sampled_negatives
            ])

            # Shuffle so clicked article isn't always first
            order = rng.permutation(5)

            groups.append(five_indices[order])
            targets.append(int(np.where(order == 0)[0][0]))

    return (
        df,
        np.asarray(groups, dtype=np.int64).reshape(-1, 5),
        np.asarray(targets, dtype=np.int64),
        skipped
    )


# Test with dummy impression data
test_df = pd.DataFrame({
    "impression_id": [101]*7 + [102]*5 + [103]*3,
    "news_id": list("ABCDEFGHIJKLMNO"),
    "label": [
        1, 1, 0, 0, 0, 0, 0,
        1, 0, 0, 0, 0,
        1, 0, 0
    ]
})

test_rows, groups, targets, skipped = build_five_way_groups(
    test_df
)

assert groups.shape == (3, 5)
assert targets.shape == (3,)
assert skipped == 1

for indices, target in zip(groups, targets):
    selected = test_rows.iloc[indices]

    assert selected["impression_id"].nunique() == 1
    assert selected["news_id"].nunique() == 5
    assert selected["label"].sum() == 1
    assert selected["label"].iloc[target] == 1

print("Training groups:", len(groups))
print("Group shape:", groups.shape)
print("Skipped groups:", skipped)
print("Clicked positions:", targets.tolist())
print("\nFive-way grouping test PASSED!")


Training groups: 3
Group shape: (3, 5)
Skipped groups: 1
Clicked positions: [2, 1, 1]

Five-way grouping test PASSED!


In [9]:

# ==========================================================
# 6. BATCHED MLP TRAINING LOOP
# ==========================================================

import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Training device:", device)


def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0
    total_groups = 0

    for (
        cat, subcat, hist_cat, hist_subcat,
        numeric, targets
    ) in loader:

        cat = cat.to(device)
        subcat = subcat.to(device)
        hist_cat = hist_cat.to(device)
        hist_subcat = hist_subcat.to(device)
        numeric = numeric.to(device)
        targets = targets.to(device)

        batch_size = cat.shape[0]
        slate_size = cat.shape[1]

        # Each candidate in a slate shares the user's history
        repeated_hist_cat = hist_cat.repeat_interleave(
            slate_size, dim=0
        )
        repeated_hist_subcat = hist_subcat.repeat_interleave(
            slate_size, dim=0
        )

        # Score all candidates
        scores = model(
            cat.reshape(-1),
            subcat.reshape(-1),
            repeated_hist_cat,
            repeated_hist_subcat,
            numeric.reshape(-1, 5)
        ).reshape(batch_size, slate_size)

        loss = five_way_loss(scores, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * batch_size
        total_groups += batch_size

    return total_loss / total_groups


# ==========================================================
# TEST USING DUMMY TRAINING GROUPS
# ==========================================================

torch.manual_seed(42)

n_groups = 20
history_length = 4

dummy_cat = torch.randint(1, 20, (n_groups, 5))
dummy_subcat = torch.randint(1, 300, (n_groups, 5))

dummy_hist_cat = torch.randint(
    0, 20, (n_groups, history_length)
)
dummy_hist_subcat = torch.randint(
    0, 300, (n_groups, history_length)
)

dummy_numeric = torch.rand(n_groups, 5, 5)
dummy_targets = torch.randint(0, 5, (n_groups,))

dummy_dataset = TensorDataset(
    dummy_cat,
    dummy_subcat,
    dummy_hist_cat,
    dummy_hist_subcat,
    dummy_numeric,
    dummy_targets
)

dummy_loader = DataLoader(
    dummy_dataset,
    batch_size=4,
    shuffle=True
)

# Fresh model for the training test
test_model = FeatureMLP(
    n_categories=20,
    n_subcategories=300
).to(device)

optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=1e-3
)

train_loss = train_one_epoch(
    test_model,
    dummy_loader,
    optimizer,
    device
)

assert np.isfinite(train_loss)

print("Training groups:", n_groups)
print("Batch size:", 4)
print("Training loss:", round(train_loss, 4))
print("\nBatched training loop PASSED!")


Training device: cuda
Training groups: 20
Batch size: 4
Training loss: 1.6244

Batched training loop PASSED!


In [10]:

# ==========================================================
# 7. TUNE AUC AND BEST MODEL CHECKPOINT
# ==========================================================

import numpy as np
import pandas as pd
import torch
from pathlib import Path


def mean_impression_auc(impression_ids, labels, scores):
    """
    Compute mean AUC across impressions.
    Impressions without both clicks and non-clicks are skipped.
    Tied scores receive average ranks.
    """

    df = pd.DataFrame({
        "impression_id": impression_ids,
        "label": labels,
        "score": scores
    })

    assert df["label"].isin([0, 1]).all()
    assert np.isfinite(df["score"]).all()

    # Rank candidate scores within each impression
    df["rank"] = df.groupby(
        "impression_id", sort=False
    )["score"].rank(method="average")

    df["positive_rank"] = df["rank"] * df["label"]

    stats = df.groupby("impression_id", sort=False).agg(
        n=("label", "size"),
        n_pos=("label", "sum"),
        pos_rank_sum=("positive_rank", "sum")
    )

    stats["n_neg"] = stats["n"] - stats["n_pos"]

    valid = stats[
        (stats["n_pos"] > 0) &
        (stats["n_neg"] > 0)
    ]

    if valid.empty:
        raise ValueError("No valid impressions for AUC.")

    aucs = (
        valid["pos_rank_sum"]
        - valid["n_pos"] * (valid["n_pos"] + 1) / 2
    ) / (valid["n_pos"] * valid["n_neg"])

    return {
        "mean_auc": float(aucs.mean()),
        "valid_impressions": len(valid),
        "skipped_impressions": len(stats) - len(valid)
    }


def save_best_checkpoint(
    model, optimizer, epoch, tune_auc, best_auc, path
):
    """Save model only when TUNE AUC improves."""

    if not np.isfinite(tune_auc):
        raise ValueError("Invalid TUNE AUC.")

    if tune_auc <= best_auc:
        return best_auc, False

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "tune_mean_auc": tune_auc,
        "architecture": "53-64-32-1"
    }, path)

    return tune_auc, True


# ==========================================================
# TEST WITH SMALL DUMMY TUNE DATA
# ==========================================================

test_impressions = ["A"] * 5 + ["B"] * 5 + ["C"] * 5

test_labels = [
    1, 0, 0, 0, 0,
    0, 1, 0, 0, 0,
    0, 0, 0, 0, 0
]

test_scores = [
    0.9, 0.1, 0.2, 0.3, 0.4,
    0.9, 0.2, 0.1, 0.4, 0.5,
    0.1, 0.2, 0.3, 0.4, 0.5
]

result = mean_impression_auc(
    test_impressions, test_labels, test_scores
)

assert np.isclose(result["mean_auc"], 0.625)
assert result["valid_impressions"] == 2
assert result["skipped_impressions"] == 1

# Test checkpoint saving in a temporary location
# This does NOT save a real trained model.
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    checkpoint_path = Path(tmp) / "test_checkpoint.pt"

    best_auc, saved = save_best_checkpoint(
        test_model,
        optimizer,
        epoch=1,
        tune_auc=result["mean_auc"],
        best_auc=-1.0,
        path=checkpoint_path
    )

    assert saved
    assert checkpoint_path.exists()

print("Mean impression AUC:", result["mean_auc"])
print("Valid impressions:", result["valid_impressions"])
print("Skipped impressions:", result["skipped_impressions"])
print("Checkpoint saving:", "PASSED")
print("\nTUNE evaluation and checkpoint test PASSED!")


Mean impression AUC: 0.625
Valid impressions: 2
Skipped impressions: 1
Checkpoint saving: PASSED

TUNE evaluation and checkpoint test PASSED!


In [11]:

# ==========================================================
# 8. PREDICT CANDIDATE RELEVANCE SCORES
# ==========================================================

import numpy as np
import pandas as pd
import torch


def predict_candidate_scores(
    model,
    category_idx,
    subcategory_idx,
    history_category_idx,
    history_subcategory_idx,
    numerical_features,
    device,
    batch_size=8192
):
    """
    Predict one relevance score per candidate article.

    All inputs must have the same number of rows.
    History arrays must be padded to equal lengths.
    """
    n = len(category_idx)

    assert len(subcategory_idx) == n
    assert len(history_category_idx) == n
    assert len(history_subcategory_idx) == n
    assert len(numerical_features) == n

    model = model.to(device)
    model.eval()

    all_scores = []

    with torch.inference_mode():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)

            cat = torch.as_tensor(
                category_idx[start:end],
                dtype=torch.long,
                device=device
            )

            subcat = torch.as_tensor(
                subcategory_idx[start:end],
                dtype=torch.long,
                device=device
            )

            hist_cat = torch.as_tensor(
                history_category_idx[start:end],
                dtype=torch.long,
                device=device
            )

            hist_subcat = torch.as_tensor(
                history_subcategory_idx[start:end],
                dtype=torch.long,
                device=device
            )

            numeric = torch.as_tensor(
                numerical_features[start:end],
                dtype=torch.float32,
                device=device
            )

            scores = model(
                cat, subcat, hist_cat, hist_subcat, numeric
            )

            all_scores.append(scores.cpu().numpy())

    return (
        np.concatenate(all_scores)
        if all_scores else np.array([], dtype=np.float32)
    )


# ==========================================================
# SMALL TEST USING EXISTING DUMMY DATA
# ==========================================================

test_scores = predict_candidate_scores(
    model=test_model,
    category_idx=dummy_cat[0].cpu().numpy(),
    subcategory_idx=dummy_subcat[0].cpu().numpy(),
    history_category_idx=np.repeat(
        dummy_hist_cat[:1].cpu().numpy(), 5, axis=0
    ),
    history_subcategory_idx=np.repeat(
        dummy_hist_subcat[:1].cpu().numpy(), 5, axis=0
    ),
    numerical_features=dummy_numeric[0].cpu().numpy(),
    device=device,
    batch_size=2
)

# Test the final score-file format
score_df = pd.DataFrame({
    "impression_id": ["dummy_1"] * 5,
    "news_id": [f"N{i}" for i in range(1, 6)],
    "score": test_scores
})

assert len(score_df) == 5
assert score_df.columns.tolist() == [
    "impression_id", "news_id", "score"
]
assert np.isfinite(score_df["score"]).all()

print(score_df)
print("\nCandidate score prediction test PASSED!")


  impression_id news_id     score
0       dummy_1      N1  0.038053
1       dummy_1      N2  0.017603
2       dummy_1      N3  0.022569
3       dummy_1      N4  0.005909
4       dummy_1      N5 -0.024181

Candidate score prediction test PASSED!


In [12]:

# ==========================================================
# 9. PREPARE FINAL MLP INPUTS
# ==========================================================

import numpy as np
import pandas as pd

# Five numerical inputs for the 53-dimensional MLP
NUMERICAL_COLUMNS = [
    "log_hist_len",
    "cat_match",
    "user_article_sim",
    "slate_exposure_pct",
    "log_article_age_hours"
]

MAX_HISTORY = 50


def prepare_mlp_inputs(
    df,
    news_category_lookup,
    news_subcategory_lookup,
    fit_mean,
    fit_std,
    max_history=MAX_HISTORY
):
    """
    Convert engineered features and click histories
    into inputs for the 53-dimensional Feature MLP.

    Numerical standardisation uses FIT statistics only.
    """

    required = [
        "category_idx",
        "subcategory_idx",
        "hist_len",
        "cat_match",
        "user_article_sim",
        "slate_exposure_pct",
        "log_article_age_hours",
        "history"
    ]

    missing = [col for col in required if col not in df.columns]

    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    n = len(df)

    # Candidate article category/subcategory indices
    category_idx = df["category_idx"].to_numpy(
        dtype=np.int64
    )

    subcategory_idx = df["subcategory_idx"].to_numpy(
        dtype=np.int64
    )

    # User click-history indices, padded to MAX_HISTORY
    hist_cat = np.zeros(
        (n, max_history), dtype=np.int64
    )

    hist_subcat = np.zeros(
        (n, max_history), dtype=np.int64
    )

    for i, history in enumerate(df["history"]):
        if not isinstance(history, str):
            continue

        # Use the most recent clicked articles
        history_ids = history.split()[-max_history:]

        for j, news_id in enumerate(history_ids):
            hist_cat[i, j] = news_category_lookup.get(
                news_id, 0
            )
            hist_subcat[i, j] = news_subcategory_lookup.get(
                news_id, 0
            )

    # Prepare the five numerical features
    numerical = np.column_stack([
        np.log1p(df["hist_len"].to_numpy(dtype=np.float32)),
        df["cat_match"].to_numpy(dtype=np.float32),
        df["user_article_sim"].to_numpy(dtype=np.float32),
        df["slate_exposure_pct"].to_numpy(dtype=np.float32),
        df["log_article_age_hours"].to_numpy(dtype=np.float32)
    ])

    # Standardise using statistics computed on FIT only
    fit_mean = np.asarray(fit_mean, dtype=np.float32).reshape(1, 5)
    fit_std = np.asarray(fit_std, dtype=np.float32).reshape(1, 5)

    if not np.all(np.isfinite(fit_mean)) or not np.all(
        np.isfinite(fit_std) & (fit_std > 0)
    ):
        raise ValueError("Invalid FIT normalisation statistics.")

    numerical = (numerical - fit_mean) / fit_std

    assert np.isfinite(numerical).all()

    return {
        "category_idx": category_idx,
        "subcategory_idx": subcategory_idx,
        "history_category_idx": hist_cat,
        "history_subcategory_idx": hist_subcat,
        "numerical_features": numerical.astype(np.float32)
    }


# ==========================================================
# SMALL INPUT-FORMAT TEST (NO REAL DATA USED)
# ==========================================================

sample_batch = pd.DataFrame({
    "category_idx": [1, 2, 3],
    "subcategory_idx": [10, 20, 30],
    "hist_len": [2, 0, 3],
    "cat_match": [0.5, 0.0, 0.33],
    "user_article_sim": [0.4, 0.0, -0.2],
    "slate_exposure_pct": [0.25, 0.5, 0.75],
    "log_article_age_hours": [1.2, 0.0, 2.0],
    "history": ["N1 N2", "", "N2 N3 N1"]
})

category_lookup = {"N1": 1, "N2": 2, "N3": 3}
subcategory_lookup = {"N1": 10, "N2": 20, "N3": 30}

# Dummy normalisation statistics only for this test
sample_inputs = prepare_mlp_inputs(
    sample_batch,
    category_lookup,
    subcategory_lookup,
    fit_mean=np.zeros(5),
    fit_std=np.ones(5)
)

assert sample_inputs["category_idx"].shape == (3,)
assert sample_inputs["subcategory_idx"].shape == (3,)
assert sample_inputs["history_category_idx"].shape == (3, 50)
assert sample_inputs["history_subcategory_idx"].shape == (3, 50)
assert sample_inputs["numerical_features"].shape == (3, 5)

# Confirm the user with no history has zero-padded inputs
assert np.all(sample_inputs["history_category_idx"][1] == 0)

print("Numerical inputs:", sample_inputs["numerical_features"].shape)
print("User history inputs:", sample_inputs["history_category_idx"].shape)
print("All input shapes correct!")
print("\nMLP input preparation PASSED!")


Numerical inputs: (3, 5)
User history inputs: (3, 50)
All input shapes correct!

MLP input preparation PASSED!


In [13]:

# ==========================================================
# 10. FIT-ONLY NUMERICAL STANDARDISATION
# ==========================================================

import numpy as np
import pyarrow.parquet as pq

RAW_NUMERIC_COLUMNS = [
    "hist_len",
    "cat_match",
    "user_article_sim",
    "slate_exposure_pct",
    "log_article_age_hours"
]


def get_numeric_matrix(df):
    """Extract the five MLP numerical features."""

    x = df[RAW_NUMERIC_COLUMNS].to_numpy(
        dtype=np.float64, copy=True
    )

    # Log-transform user history length
    x[:, 0] = np.log1p(x[:, 0])

    if not np.isfinite(x).all():
        raise ValueError("Invalid numerical feature values.")

    return x


def calculate_fit_stats(batches):
    """Calculate FIT mean/std without loading all rows."""

    count = 0
    mean = np.zeros(5, dtype=np.float64)
    m2 = np.zeros(5, dtype=np.float64)

    for df in batches:
        x = get_numeric_matrix(df)

        if len(x) == 0:
            continue

        batch_count = len(x)
        batch_mean = x.mean(axis=0)
        batch_m2 = ((x - batch_mean) ** 2).sum(axis=0)

        new_count = count + batch_count
        delta = batch_mean - mean

        mean += delta * batch_count / new_count
        m2 += batch_m2 + (
            delta ** 2 * count * batch_count / new_count
        )

        count = new_count

    if count == 0:
        raise ValueError("No FIT observations found.")

    std = np.sqrt(m2 / count)

    # Prevent division by zero for constant features
    std[std < 1e-8] = 1.0

    return mean.astype(np.float32), std.astype(np.float32)


def load_fit_stats(feature_path, batch_size=100_000):
    """Stream updated FIT features in manageable batches."""

    pf = pq.ParquetFile(feature_path)

    batches = (
        batch.to_pandas()
        for batch in pf.iter_batches(
            batch_size=batch_size,
            columns=RAW_NUMERIC_COLUMNS
        )
    )

    return calculate_fit_stats(batches)


# ==========================================================
# TEST USING DUMMY DATA ONLY
# ==========================================================

test_mean, test_std = calculate_fit_stats([sample_batch])

assert test_mean.shape == (5,)
assert test_std.shape == (5,)
assert np.isfinite(test_mean).all()
assert (test_std > 0).all()

print("FIT mean shape:", test_mean.shape)
print("FIT std shape:", test_std.shape)
print("Standardisation test PASSED!")


FIT mean shape: (5,)
FIT std shape: (5,)
Standardisation test PASSED!


In [ ]:
import config
from pathlib import Path

print("Project:", config.PROJECT_DIR)
print("Data:", config.DATA_DIR)
print("Processed:", config.PROCESSED_DIR)

print("\nProcessed folder exists:", Path(config.PROCESSED_DIR).exists())

print("\nAvailable Parquet files:")

for file in Path(config.PROCESSED_DIR).glob("*.parquet"):
    print(file.name)

Project: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project
Data: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data
Processed: G:\.shortcut-targets-by-id\1D4Db1s7DJ5PVK9CFiV-pklpCW5dr1uOM\BT4222_Project\data\processed

Processed folder exists: True

Available Parquet files:
tune_impressions.parquet
news_metadata.parquet
fit_impressions.parquet
eval_impressions.parquet
tune_candidates.parquet
eval_candidates.parquet
fit_candidates_sampled.parquet


In [ ]:
import numpy as np
import pandas as pd

from utils.data_utils import (
    load_news,
    load_behaviors,
    explode_impressions
)

from utils.metrics import (
    evaluate,
    evaluate_by_segment,
    topk,
    topk_diversity
)

print("All imports successful!")

All imports successful!


In [ ]:
from pathlib import Path
import pandas as pd
import config

processed_dir = Path(config.PROCESSED_DIR)

fit_candidates = pd.read_parquet(
    processed_dir / "fit_candidates_sampled.parquet"
)

tune_candidates = pd.read_parquet(
    processed_dir / "tune_candidates.parquet"
)

eval_candidates = pd.read_parquet(
    processed_dir / "eval_candidates.parquet"
)

news_metadata = pd.read_parquet(
    processed_dir / "news_metadata.parquet"
)

print("FIT:", fit_candidates.shape)
print("TUNE:", tune_candidates.shape)
print("EVAL:", eval_candidates.shape)
print("NEWS:", news_metadata.shape)

FIT: (12894210, 11)
TUNE: (17400106, 11)
EVAL: (14085557, 11)
NEWS: (104151, 9)


In [ ]:
print("FIT COLUMNS:")
print(fit_candidates.columns.tolist())

print("\nNEWS COLUMNS:")
print(news_metadata.columns.tolist())

FIT COLUMNS:
['impression_id', 'user_id', 'user_idx', 'time', 'news_id', 'news_idx', 'label', 'prior_exposure_count', 'cold_article', 'category', 'subcategory']

NEWS COLUMNS:
['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities', 'news_idx']


In [ ]:
print("=== FIT CANDIDATES ===")
fit_candidates.info()

print("\n=== MISSING VALUES ===")
print(fit_candidates.isna().sum())

print("\n=== LABEL DISTRIBUTION ===")
print(fit_candidates["label"].value_counts())

print("\n=== COLD ARTICLE DISTRIBUTION ===")
print(fit_candidates["cold_article"].value_counts())

print("\n=== EXPOSURE COUNT ===")
print(fit_candidates["prior_exposure_count"].describe())

=== FIT CANDIDATES ===
<class 'pandas.DataFrame'>
RangeIndex: 12894210 entries, 0 to 12894209
Data columns (total 11 columns):
 #   Column                Dtype         
---  ------                -----         
 0   impression_id         str           
 1   user_id               str           
 2   user_idx              int64         
 3   time                  datetime64[ns]
 4   news_id               str           
 5   news_idx              int64         
 6   label                 int64         
 7   prior_exposure_count  int64         
 8   cold_article          int64         
 9   category              str           
 10  subcategory           str           
dtypes: datetime64[ns](1), int64(5), str(5)
memory usage: 1.5 GB

=== MISSING VALUES ===
impression_id           0
user_id                 0
user_idx                0
time                    0
news_id                 0
news_idx                0
label                   0
prior_exposure_count    0
cold_article            0
cate

In [ ]:
print("=== LABEL ===")
print(fit_candidates["label"].value_counts().sort_index())
print()

print("=== COLD ARTICLE ===")
print(fit_candidates["cold_article"].value_counts().sort_index())
print()

print("=== CATEGORY ===")
print(fit_candidates["category"].value_counts().head(20))
print()

print("=== SUBCATEGORY ===")
print(fit_candidates["subcategory"].value_counts().head(20))
print()

print("=== MISSING VALUES ===")
print(fit_candidates.isna().sum())

=== LABEL ===
label
0    10181598
1     2712612
Name: count, dtype: int64

=== COLD ARTICLE ===
cold_article
0    8604602
1    4289608
Name: count, dtype: int64

=== CATEGORY ===
category
news             3912045
sports           1474967
lifestyle        1343340
finance          1237195
entertainment     744890
music             694276
tv                632264
foodanddrink      615947
health            575552
travel            525697
autos             465798
movies            257434
weather           227391
video             186788
kids                 562
northamerica          64
Name: count, dtype: int64

=== SUBCATEGORY ===
subcategory
newsus                     1247918
newsworld                   754560
football_nfl                717989
newscrime                   590320
newspolitics                580683
finance-companies           472687
celebrity                   320918
music-celebrity             314367
lifestylebuzz               308544
lifestyleroyals             298873
tv-

In [ ]:
# Create a compact feature table for the MLP

mlp_train = fit_candidates[
    [
        "user_idx",
        "news_idx",
        "prior_exposure_count",
        "cold_article",
        "category",
        "subcategory",
        "time",
        "label",
    ]
].copy()

# Time-based features
mlp_train["hour"] = mlp_train["time"].dt.hour
mlp_train["day_of_week"] = mlp_train["time"].dt.dayofweek

# We no longer need the original timestamp for the first MLP
mlp_train.drop(columns=["time"], inplace=True)

print("MLP feature table shape:", mlp_train.shape)
print("\nColumns:")
print(mlp_train.columns.tolist())

print("\nFirst 5 rows:")
display(mlp_train.head())

C:\Users\eishw\AppData\Local\Temp\ipykernel_14616\2164718650.py:18: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  mlp_train["day_of_week"] = mlp_train["time"].dt.dayofweek


MLP feature table shape: (12894210, 9)

Columns:
['user_idx', 'news_idx', 'prior_exposure_count', 'cold_article', 'category', 'subcategory', 'label', 'hour', 'day_of_week']

First 5 rows:


C:\Users\eishw\AppData\Local\Temp\ipykernel_14616\2164718650.py:21: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  mlp_train.drop(columns=["time"], inplace=True)


,user_idx,news_idx,prior_exposure_count,cold_article,category,subcategory,label,hour,day_of_week
0,503854,88046,0,1,sports,basketball_ncaa,1,1,5
1,503854,100229,0,1,travel,travelnews,0,1,5
2,503854,79136,0,1,autos,autosenthusiasts,0,1,5
3,503854,94865,0,1,lifestyle,lifestylebuzz,0,1,5
4,503854,81953,0,1,news,newspolitics,0,1,5


In [ ]:
def create_mlp_features(df):
    features = df[
        [
            "user_idx",
            "news_idx",
            "prior_exposure_count",
            "cold_article",
            "category",
            "subcategory",
            "time",
            "label",
        ]
    ].copy()

    # Time features
    features["hour"] = features["time"].dt.hour
    features["day_of_week"] = features["time"].dt.day_of_week

    # Remove original timestamp
    features = features.drop(columns=["time"])

    return features


# Create features for all three splits
mlp_train = create_mlp_features(fit_candidates)
mlp_tune = create_mlp_features(tune_candidates)
mlp_eval = create_mlp_features(eval_candidates)


print("FIT:", mlp_train.shape)
print("TUNE:", mlp_tune.shape)
print("EVAL:", mlp_eval.shape)

print("\nColumns match:")
print(mlp_train.columns.tolist() == mlp_tune.columns.tolist())
print(mlp_train.columns.tolist() == mlp_eval.columns.tolist())

FIT: (12894210, 9)
TUNE: (17400106, 9)
EVAL: (14085557, 9)

Columns match:
True
True


In [ ]:
print("=== ID / CATEGORY SIZES ===")

print("Number of users:", fit_candidates["user_idx"].nunique())
print("Maximum user_idx:", fit_candidates["user_idx"].max())

print("\nNumber of news articles:", fit_candidates["news_idx"].nunique())
print("Maximum news_idx:", fit_candidates["news_idx"].max())

print("\nNumber of categories:", fit_candidates["category"].nunique())
print("Categories:", fit_candidates["category"].unique())

print("\nNumber of subcategories:", fit_candidates["subcategory"].nunique())
print("Subcategories:", fit_candidates["subcategory"].unique())

=== ID / CATEGORY SIZES ===
Number of users: 654870
Maximum user_idx: 711222

Number of news articles: 19090
Maximum news_idx: 101526

Number of categories: 16
Categories: <ArrowStringArray>
[       'sports',        'travel',         'autos',     'lifestyle',
          'news',       'finance',         'music',            'tv',
  'foodanddrink',        'movies', 'entertainment',        'health',
         'video',       'weather',          'kids',  'northamerica']
Length: 16, dtype: str

Number of subcategories: 218
Subcategories: <ArrowStringArray>
[       'basketball_ncaa',             'travelnews',       'autosenthusiasts',
          'lifestylebuzz',           'newspolitics',      'finance-companies',
              'musicnews',              'newsworld',           'tv-celebrity',
     'restaurantsandnews',
 ...
            'smartliving',              'cocktails',     'northamerica-video',
           'mentalhealth', 'tv-golden-globes-video',   'spendingandborrowing',
               'hol

In [ ]:
print("=== USER ID RANGES ===")

for name, df in [
    ("FIT", fit_candidates),
    ("TUNE", tune_candidates),
    ("EVAL", eval_candidates),
]:
    print(
        f"{name}: "
        f"min={df['user_idx'].min()}, "
        f"max={df['user_idx'].max()}, "
        f"unique={df['user_idx'].nunique():,}"
    )


print("\n=== NEWS ID RANGES ===")

for name, df in [
    ("FIT", fit_candidates),
    ("TUNE", tune_candidates),
    ("EVAL", eval_candidates),
]:
    print(
        f"{name}: "
        f"min={df['news_idx'].min()}, "
        f"max={df['news_idx'].max()}, "
        f"unique={df['news_idx'].nunique():,}"
    )

=== USER ID RANGES ===
FIT: min=1, max=711222, unique=654,870
TUNE: min=2, max=711221, unique=286,814
EVAL: min=0, max=711222, unique=216,779

=== NEWS ID RANGES ===
FIT: min=17, max=101526, unique=19,090
TUNE: min=7, max=101447, unique=9,107
EVAL: min=7, max=104150, unique=6,997


In [ ]:
# Global ID ranges across all three splits

all_user_max = max(
    fit_candidates["user_idx"].max(),
    tune_candidates["user_idx"].max(),
    eval_candidates["user_idx"].max(),
)

all_news_max = max(
    fit_candidates["news_idx"].max(),
    tune_candidates["news_idx"].max(),
    eval_candidates["news_idx"].max(),
)

print("Global user_idx range:")
print("Minimum:", min(
    fit_candidates["user_idx"].min(),
    tune_candidates["user_idx"].min(),
    eval_candidates["user_idx"].min()
))
print("Maximum:", all_user_max)

print("\nGlobal news_idx range:")
print("Minimum:", min(
    fit_candidates["news_idx"].min(),
    tune_candidates["news_idx"].min(),
    eval_candidates["news_idx"].min()
))
print("Maximum:", all_news_max)

print("\nEmbedding sizes we will need:")
print("User embedding size:", all_user_max + 1)
print("News embedding size:", all_news_max + 1)

Global user_idx range:
Minimum: 0
Maximum: 711222

Global news_idx range:
Minimum: 7
Maximum: 104150

Embedding sizes we will need:
User embedding size: 711223
News embedding size: 104151


In [ ]:
# Build category vocabulary from all splits

all_categories = pd.concat([
    fit_candidates["category"],
    tune_candidates["category"],
    eval_candidates["category"]
]).astype(str).unique()

all_subcategories = pd.concat([
    fit_candidates["subcategory"],
    tune_candidates["subcategory"],
    eval_candidates["subcategory"]
]).astype(str).unique()

# Sort so the mapping is deterministic
all_categories = sorted(all_categories)
all_subcategories = sorted(all_subcategories)

category_to_idx = {
    category: idx
    for idx, category in enumerate(all_categories)
}

subcategory_to_idx = {
    subcategory: idx
    for idx, subcategory in enumerate(all_subcategories)
}

print("Number of categories:", len(category_to_idx))
print("Number of subcategories:", len(subcategory_to_idx))

print("\nCategory mapping:")
print(category_to_idx)

print("\nFirst 20 subcategory mappings:")
for subcategory, idx in list(subcategory_to_idx.items())[:20]:
    print(idx, "→", subcategory)

Number of categories: 16
Number of subcategories: 230

Category mapping:
{'autos': 0, 'entertainment': 1, 'finance': 2, 'foodanddrink': 3, 'health': 4, 'kids': 5, 'lifestyle': 6, 'movies': 7, 'music': 8, 'news': 9, 'northamerica': 10, 'sports': 11, 'travel': 12, 'tv': 13, 'video': 14, 'weather': 15}

First 20 subcategory mappings:
0 → ads-latingrammys
1 → ads-lung-health
2 → advice
3 → animals
4 → autosbuying
5 → autosclassics
6 → autoscompact
7 → autosenthusiasts
8 → autoshybrids
9 → autoslosangeles
10 → autosluxury
11 → autosmidsize
12 → autosmotorcycles
13 → autosnews
14 → autosownership
15 → autospassenger
16 → autosresearch
17 → autosresearchguides
18 → autosreview
19 → autossema


In [ ]:
#check
print("All FIT categories mapped:",
      fit_candidates["category"].astype(str).isin(category_to_idx).all())

print("All TUNE categories mapped:",
      tune_candidates["category"].astype(str).isin(category_to_idx).all())

print("All EVAL categories mapped:",
      eval_candidates["category"].astype(str).isin(category_to_idx).all())

print("\nAll FIT subcategories mapped:",
      fit_candidates["subcategory"].astype(str).isin(subcategory_to_idx).all())

print("All TUNE subcategories mapped:",
      tune_candidates["subcategory"].astype(str).isin(subcategory_to_idx).all())

print("All EVAL subcategories mapped:",
      eval_candidates["subcategory"].astype(str).isin(subcategory_to_idx).all())

All FIT categories mapped: True
All TUNE categories mapped: True
All EVAL categories mapped: True

All FIT subcategories mapped: True
All TUNE subcategories mapped: True
All EVAL subcategories mapped: True


In [ ]:
import torch

print("PyTorch version:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

PyTorch version: 2.15.0+cpu
CUDA available: False
Using CPU


In [ ]:
#checking available RAM
import psutil

memory = psutil.virtual_memory()

print("Total RAM:", round(memory.total / (1024**3), 2), "GB")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("Currently used:", round(memory.percent, 2), "%")

Total RAM: 15.28 GB
Available RAM: 0.71 GB
Currently used: 95.4 %


In [ ]:
import sys
import gc

variables = {
    "fit_candidates": fit_candidates,
    "tune_candidates": tune_candidates,
    "eval_candidates": eval_candidates,
    "news_metadata": news_metadata,
    "mlp_train": mlp_train,
    "mlp_tune": mlp_tune,
    "mlp_eval": mlp_eval,
}

print("=== DATAFRAME MEMORY USAGE ===")

for name, df in variables.items():
    memory_gb = df.memory_usage(deep=True).sum() / (1024**3)
    print(f"{name}: {memory_gb:.2f} GB")

=== DATAFRAME MEMORY USAGE ===
fit_candidates: 1.51 GB
tune_candidates: 2.03 GB
eval_candidates: 1.63 GB
news_metadata: 0.09 GB
mlp_train: 0.98 GB
mlp_tune: 1.33 GB
mlp_eval: 1.07 GB


In [ ]:
# Remove the duplicate feature DataFrames
del mlp_train
del mlp_tune
del mlp_eval

# Ask Python to release unused memory
import gc
gc.collect()

# Check available RAM again
import psutil

memory = psutil.virtual_memory()

print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("RAM usage:", memory.percent, "%")

Available RAM: 0.55 GB
RAM usage: 96.4 %


In [ ]:
import pyarrow.parquet as pq
from pathlib import Path

processed_dir = Path(config.PROCESSED_DIR)

files = [
    "fit_candidates_sampled.parquet",
    "tune_candidates.parquet",
    "eval_candidates.parquet",
]

for filename in files:
    file_path = processed_dir / filename

    parquet_file = pq.ParquetFile(file_path)

    print(f"\n{filename}")
    print("Rows:", f"{parquet_file.metadata.num_rows:,}")
    print("Row groups:", parquet_file.metadata.num_row_groups)
    print("Columns:", parquet_file.schema.names)


fit_candidates_sampled.parquet
Rows: 12,894,210
Row groups: 133
Columns: ['impression_id', 'user_id', 'user_idx', 'time', 'news_id', 'news_idx', 'label', 'prior_exposure_count', 'cold_article', 'category', 'subcategory']

tune_candidates.parquet
Rows: 17,400,106
Row groups: 35
Columns: ['impression_id', 'user_id', 'user_idx', 'time', 'news_id', 'news_idx', 'label', 'prior_exposure_count', 'cold_article', 'category', 'subcategory']

eval_candidates.parquet
Rows: 14,085,557
Row groups: 29
Columns: ['impression_id', 'user_id', 'user_idx', 'time', 'news_id', 'news_idx', 'label', 'prior_exposure_count', 'cold_article', 'category', 'subcategory']


In [ ]:
import gc
import psutil

# Remove the large DataFrames from memory
del fit_candidates
del tune_candidates
del eval_candidates

# Release unused memory
gc.collect()

# Check available RAM
memory = psutil.virtual_memory()

print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("RAM usage:", memory.percent, "%")

Available RAM: 0.54 GB
RAM usage: 96.5 %


In [ ]:
# ============================================================
# Read the FIT dataset in small batches
# ============================================================

from pathlib import Path
import pyarrow.parquet as pq
import gc

# Locate the processed FIT dataset
processed_dir = Path(config.PROCESSED_DIR)
fit_path = processed_dir / "fit_candidates_sampled.parquet"

# Select only the columns needed by our MLP
FEATURE_COLUMNS = [
    "user_idx",
    "news_idx",
    "prior_exposure_count",
    "cold_article",
    "category",
    "subcategory",
    "time",
    "label"
]

# Open the Parquet file without loading the entire dataset
fit_parquet = pq.ParquetFile(fit_path)

# Create a batch iterator
fit_batch_iterator = fit_parquet.iter_batches(
    batch_size=8192,
    columns=FEATURE_COLUMNS
)

# Read just the first batch
first_batch = next(fit_batch_iterator)

# Convert this batch into a Pandas DataFrame
first_batch_df = first_batch.to_pandas()

print("FIT batch successfully loaded!")
print("Batch shape:", first_batch_df.shape)
print("\nColumns:")
print(first_batch_df.columns.tolist())

print("\nFirst 5 rows:")
display(first_batch_df.head())

print("\nMemory used by this batch:")
print(
    round(first_batch_df.memory_usage(deep=True).sum() / (1024**2), 2),
    "MB"
)

FIT batch successfully loaded!
Batch shape: (8192, 8)

Columns:
['user_idx', 'news_idx', 'prior_exposure_count', 'cold_article', 'category', 'subcategory', 'time', 'label']

First 5 rows:


,user_idx,news_idx,prior_exposure_count,cold_article,category,subcategory,time,label
0,503854,88046,0,1,sports,basketball_ncaa,2019-11-09 01:25:34,1
1,503854,100229,0,1,travel,travelnews,2019-11-09 01:25:34,0
2,503854,79136,0,1,autos,autosenthusiasts,2019-11-09 01:25:34,0
3,503854,94865,0,1,lifestyle,lifestylebuzz,2019-11-09 01:25:34,0
4,503854,81953,0,1,news,newspolitics,2019-11-09 01:25:34,0



Memory used by this batch:
0.64 MB


In [ ]:
# ============================================================
# Create categorical encoding dictionaries
# ============================================================

# Build vocabularies from the available article metadata.
# These are descriptive article attributes, not target labels.

category_values = sorted(
    news_metadata["category"].dropna().astype(str).unique()
)

subcategory_values = sorted(
    news_metadata["subcategory"].dropna().astype(str).unique()
)

# Reserve index 0 for unknown categories
category_to_idx = {
    category: idx + 1
    for idx, category in enumerate(category_values)
}

subcategory_to_idx = {
    subcategory: idx + 1
    for idx, subcategory in enumerate(subcategory_values)
}

# Number of embedding indices, including unknown index 0
NUM_CATEGORIES = len(category_to_idx) + 1
NUM_SUBCATEGORIES = len(subcategory_to_idx) + 1

print("Categorical encoding created successfully!")
print("Number of categories:", len(category_to_idx))
print("Number of subcategories:", len(subcategory_to_idx))

print("\nCategory examples:")
print(list(category_to_idx.items())[:5])

print("\nSubcategory examples:")
print(list(subcategory_to_idx.items())[:5])

print("\nEmbedding vocabulary sizes:")
print("Categories:", NUM_CATEGORIES)
print("Subcategories:", NUM_SUBCATEGORIES)

Categorical encoding created successfully!
Number of categories: 18
Number of subcategories: 285

Category examples:
[('autos', 1), ('entertainment', 2), ('finance', 3), ('foodanddrink', 4), ('games', 5)]

Subcategory examples:
[('ads-latingrammys', 1), ('ads-lung-health', 2), ('advice', 3), ('animals', 4), ('autosbuying', 5)]

Embedding vocabulary sizes:
Categories: 19
Subcategories: 286


In [ ]:
# ============================================================
# 3.3.2 — Build FIT-only user and article vocabularies
# ============================================================

import numpy as np

known_user_ids = set()
known_news_ids = set()

# Scan only the ID columns from the FIT Parquet file
for batch in fit_parquet.iter_batches(
    batch_size=65536,
    columns=["user_idx", "news_idx"]
):
    known_user_ids.update(batch.column("user_idx").to_pylist())
    known_news_ids.update(batch.column("news_idx").to_pylist())

# Convert to sorted NumPy arrays for efficient membership checks
known_user_ids = np.array(sorted(known_user_ids), dtype=np.int64)
known_news_ids = np.array(sorted(known_news_ids), dtype=np.int64)

# Embedding table sizes use the original global indices.
# Index 0 is reserved for unknown users/articles.
NUM_USERS = int(known_user_ids.max()) + 1
NUM_NEWS = int(known_news_ids.max()) + 1

print("FIT-only ID vocabularies created!")
print("Unique FIT users:", len(known_user_ids))
print("Unique FIT articles:", len(known_news_ids))

print("\nEmbedding table sizes:")
print("User embedding:", NUM_USERS)
print("News embedding:", NUM_NEWS)

print("\nReserved unknown index: 0")

FIT-only ID vocabularies created!
Unique FIT users: 654870
Unique FIT articles: 19090

Embedding table sizes:
User embedding: 711223
News embedding: 101527

Reserved unknown index: 0


In [ ]:
# ============================================================
# Feature Transformation Function
# ============================================================

import numpy as np
import pandas as pd
import torch

def transform_batch(batch_df):

    # User and article IDs
    user_ids = batch_df["user_idx"].to_numpy(dtype=np.int64).copy()
    news_ids = batch_df["news_idx"].to_numpy(dtype=np.int64).copy()

    # Map IDs not seen in FIT to unknown index 0
    user_ids[~np.isin(user_ids, known_user_ids)] = 0
    news_ids[~np.isin(news_ids, known_news_ids)] = 0

    # Encode categories
    category_ids = (
        batch_df["category"]
        .astype(str)
        .map(category_to_idx)
        .fillna(0)
        .to_numpy(dtype=np.int64)
    )

    subcategory_ids = (
        batch_df["subcategory"]
        .astype(str)
        .map(subcategory_to_idx)
        .fillna(0)
        .to_numpy(dtype=np.int64)
    )

    # Transform numerical features
    exposure = np.log1p(
        batch_df["prior_exposure_count"].to_numpy(dtype=np.float32)
    )

    cold_article = batch_df["cold_article"].to_numpy(dtype=np.float32)

    # Extract timestamp information
    timestamps = pd.to_datetime(batch_df["time"])

    hour = timestamps.dt.hour.to_numpy(dtype=np.float32)
    day = timestamps.dt.day_of_week.to_numpy(dtype=np.float32)

    # Cyclical time encoding
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)

    day_sin = np.sin(2 * np.pi * day / 7)
    day_cos = np.cos(2 * np.pi * day / 7)

    numerical_features = np.column_stack([
        exposure,
        cold_article,
        hour_sin,
        hour_cos,
        day_sin,
        day_cos
    ]).astype(np.float32)

    # Convert everything into PyTorch tensors
    return {
        "user_ids": torch.tensor(user_ids, dtype=torch.long),
        "news_ids": torch.tensor(news_ids, dtype=torch.long),
        "category_ids": torch.tensor(category_ids, dtype=torch.long),
        "subcategory_ids": torch.tensor(subcategory_ids, dtype=torch.long),
        "numerical_features": torch.tensor(
            numerical_features, dtype=torch.float32
        ),
        "labels": torch.tensor(
            batch_df["label"].to_numpy(dtype=np.float32),
            dtype=torch.float32
        )
    }

print("Feature transformation function defined successfully!")

Feature transformation function defined successfully!


In [ ]:
first_batch_tensors = transform_batch(first_batch_df)

for name, tensor in first_batch_tensors.items():
    print(f"{name}: {tuple(tensor.shape)}, {tensor.dtype}")

user_ids: (8192,), torch.int64
news_ids: (8192,), torch.int64
category_ids: (8192,), torch.int64
subcategory_ids: (8192,), torch.int64
numerical_features: (8192, 6), torch.float32
labels: (8192,), torch.float32


In [ ]:
# ============================================================
# Feature-Based MLP: Model Architecture
# ============================================================

import torch
import torch.nn as nn

class FeatureMLP(nn.Module):

    def __init__(
        self,
        num_users,
        num_news,
        num_categories,
        num_subcategories
    ):
        super().__init__()

        # Embedding layers
        self.user_embedding = nn.Embedding(
            num_users, 32, padding_idx=0
        )

        self.news_embedding = nn.Embedding(
            num_news, 32, padding_idx=0
        )

        self.category_embedding = nn.Embedding(
            num_categories, 8, padding_idx=0
        )

        self.subcategory_embedding = nn.Embedding(
            num_subcategories, 16, padding_idx=0
        )

        # Total input dimension:
        # 32 + 32 + 8 + 16 + 6 = 94
        input_dim = 94

        # Fully connected neural network
        self.mlp = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(
        self,
        user_ids,
        news_ids,
        category_ids,
        subcategory_ids,
        numerical_features
    ):

        # Convert IDs into embedding vectors
        user_vector = self.user_embedding(user_ids)
        news_vector = self.news_embedding(news_ids)

        category_vector = self.category_embedding(category_ids)
        subcategory_vector = self.subcategory_embedding(
            subcategory_ids
        )

        # Combine all feature representations
        combined_features = torch.cat(
            [
                user_vector,
                news_vector,
                category_vector,
                subcategory_vector,
                numerical_features
            ],
            dim=1
        )

        # Generate click prediction logits
        logits = self.mlp(combined_features)

        return logits.squeeze(1)


print("FeatureMLP architecture defined successfully!")

FeatureMLP architecture defined successfully!


In [ ]:
# ============================================================
# Initialise the MLP
# ============================================================

device = torch.device("cpu")

model = FeatureMLP(
    num_users=NUM_USERS,
    num_news=NUM_NEWS,
    num_categories=NUM_CATEGORIES,
    num_subcategories=NUM_SUBCATEGORIES
).to(device)

# Count trainable parameters
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model successfully initialised!")
print("Device:", device)
print("Total trainable parameters:", f"{total_parameters:,}")

print("\nModel architecture:")
print(model)

Model successfully initialised!
Device: cpu
Total trainable parameters: 26,033,209

Model architecture:
FeatureMLP(
  (user_embedding): Embedding(711223, 32, padding_idx=0)
  (news_embedding): Embedding(101527, 32, padding_idx=0)
  (category_embedding): Embedding(19, 8, padding_idx=0)
  (subcategory_embedding): Embedding(286, 16, padding_idx=0)
  (mlp): Sequential(
    (0): Linear(in_features=94, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [ ]:
# ============================================================
# Test MLP Forward Pass
# ============================================================

model.eval()

with torch.no_grad():

    logits = model(
        first_batch_tensors["user_ids"],
        first_batch_tensors["news_ids"],
        first_batch_tensors["category_ids"],
        first_batch_tensors["subcategory_ids"],
        first_batch_tensors["numerical_features"]
    )

    probabilities = torch.sigmoid(logits)

print("Forward pass successful!")

print("\nLogits shape:", logits.shape)
print("Probability shape:", probabilities.shape)

print("\nFirst 10 predicted click probabilities:")
print(probabilities[:10])

print("\nPredicted probability range:")
print(
    f"Minimum: {probabilities.min().item():.4f}",
    f"Maximum: {probabilities.max().item():.4f}"
)

Forward pass successful!

Logits shape: torch.Size([8192])
Probability shape: torch.Size([8192])

First 10 predicted click probabilities:
tensor([0.4865, 0.5150, 0.5075, 0.5286, 0.5083, 0.4989, 0.4977, 0.4894, 0.4820,
        0.5044])

Predicted probability range:
Minimum: 0.4446 Maximum: 0.5705


In [ ]:
# ============================================================
# Training Configuration
# ============================================================

LEARNING_RATE = 0.001
BATCH_SIZE = 2048
READ_BATCH_SIZE = 16384

# Binary classification loss
criterion = nn.BCEWithLogitsLoss()

# Optimiser
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print("Training configuration created!")
print("Learning rate:", LEARNING_RATE)
print("Training batch size:", BATCH_SIZE)
print("Parquet read batch size:", READ_BATCH_SIZE)
print("Loss function:", criterion)
print("Optimiser:", optimizer.__class__.__name__)

Training configuration created!
Learning rate: 0.001
Training batch size: 2048
Parquet read batch size: 16384
Loss function: BCEWithLogitsLoss()
Optimiser: Adam


In [ ]:
# ============================================================
# Streaming FIT Data Generator
# ============================================================

def iter_training_batches(
    parquet_file,
    batch_size=BATCH_SIZE,
    read_batch_size=READ_BATCH_SIZE,
    seed=42
):

    rng = np.random.default_rng(seed)

    # Randomise the order of Parquet row groups
    row_group_order = rng.permutation(
        parquet_file.num_row_groups
    ).tolist()

    # Read the FIT dataset in manageable chunks
    for arrow_batch in parquet_file.iter_batches(
        batch_size=read_batch_size,
        columns=FEATURE_COLUMNS,
        row_groups=row_group_order
    ):

        batch_df = arrow_batch.to_pandas()

        # Transform into numerical PyTorch tensors
        tensors = transform_batch(batch_df)

        # Shuffle rows within this chunk
        permutation = torch.randperm(
            len(tensors["labels"])
        )

        # Yield smaller training batches
        for start in range(0, len(permutation), batch_size):

            indices = permutation[start:start + batch_size]

            mini_batch = {
                name: tensor.index_select(0, indices)
                for name, tensor in tensors.items()
            }

            yield mini_batch

        del batch_df
        del tensors
        del arrow_batch

    print("Completed one pass through the FIT dataset.")

In [ ]:
# ============================================================
# Test Streaming Training Generator
# ============================================================

train_batch_iterator = iter_training_batches(
    parquet_file=fit_parquet,
    batch_size=BATCH_SIZE,
    read_batch_size=READ_BATCH_SIZE
)

sample_train_batch = next(train_batch_iterator)

print("Streaming generator working!\n")

for name, tensor in sample_train_batch.items():
    print(
        f"{name}: shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}"
    )

print("\nNumber of candidates in this batch:",
      len(sample_train_batch["labels"]))

print("Positive labels:",
      int(sample_train_batch["labels"].sum().item()))

print("Negative labels:",
      int((sample_train_batch["labels"] == 0).sum().item()))

Streaming generator working!

user_ids: shape=(2048,), dtype=torch.int64
news_ids: shape=(2048,), dtype=torch.int64
category_ids: shape=(2048,), dtype=torch.int64
subcategory_ids: shape=(2048,), dtype=torch.int64
numerical_features: shape=(2048, 6), dtype=torch.float32
labels: shape=(2048,), dtype=torch.float32

Number of candidates in this batch: 2048
Positive labels: 422
Negative labels: 1626


In [ ]:
# ============================================================
# MLP Training: Initial Smoke Test
# ============================================================

import gc
import time

# Release the earlier test batches and tensors
del train_batch_iterator
del sample_train_batch
del first_batch_tensors
del first_batch_df
del first_batch
gc.collect()

# Set the model to training mode
model.train()

# Prepare a fresh streaming iterator
smoke_test_iterator = iter_training_batches(
    parquet_file=fit_parquet,
    batch_size=BATCH_SIZE,
    read_batch_size=READ_BATCH_SIZE,
    seed=42
)

# Train on only 5 mini-batches
SMOKE_TEST_BATCHES = 5

smoke_test_losses = []
start_time = time.time()

for batch_number, batch in enumerate(
    smoke_test_iterator,
    start=1
):

    # Reset gradients
    optimizer.zero_grad(set_to_none=True)

    # Forward pass
    logits = model(
        batch["user_ids"],
        batch["news_ids"],
        batch["category_ids"],
        batch["subcategory_ids"],
        batch["numerical_features"]
    )

    # Calculate prediction loss
    loss = criterion(logits, batch["labels"])

    # Backpropagation
    loss.backward()

    # Update model weights
    optimizer.step()

    # Record loss
    smoke_test_losses.append(loss.item())

    print(
        f"Batch {batch_number}/{SMOKE_TEST_BATCHES} "
        f"| Loss: {loss.item():.4f}"
    )

    del batch, logits, loss

    if batch_number >= SMOKE_TEST_BATCHES:
        break

elapsed_time = time.time() - start_time

print("\nSmoke test completed!")
print("Batches trained:", len(smoke_test_losses))
print("Candidates processed:", len(smoke_test_losses) * BATCH_SIZE)
print("Average training loss:", round(
    np.mean(smoke_test_losses), 4
))
print("Time taken:", round(elapsed_time, 2), "seconds")

Batch 1/5 | Loss: 0.7087
Batch 2/5 | Loss: 0.6807
Batch 3/5 | Loss: 0.6570
Batch 4/5 | Loss: 0.6370
Batch 5/5 | Loss: 0.6209

Smoke test completed!
Batches trained: 5
Candidates processed: 10240
Average training loss: 0.6609
Time taken: 1.23 seconds


In [ ]:
# ============================================================
# Full FIT Training — One Epoch
# ============================================================

import os
import gc
import time
from pathlib import Path

# Release the partially consumed smoke-test generator
del smoke_test_iterator
gc.collect()

# Create a local checkpoint directory
checkpoint_dir = Path(PROJECT_ROOT) / "outputs" / "feature_mlp"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "feature_mlp_latest.pt"
temp_checkpoint_path = checkpoint_dir / "feature_mlp_temp.pt"

# Training settings
EPOCHS = 1
SAVE_EVERY = 1000
LOG_EVERY = 250

# Continue training from the existing smoke-test weights
model.train()

epoch_losses = []
running_loss = 0.0
running_batches = 0
total_batches = 0

start_time = time.time()

print("Starting full FIT training...")
print("Training candidates:", fit_parquet.metadata.num_rows)
print("Batch size:", BATCH_SIZE)
print("Checkpoint location:", checkpoint_path)
print("-" * 55)

for epoch in range(EPOCHS):

    train_iterator = iter_training_batches(
        parquet_file=fit_parquet,
        batch_size=BATCH_SIZE,
        read_batch_size=READ_BATCH_SIZE,
        seed=42 + epoch
    )

    for batch_number, batch in enumerate(train_iterator, start=1):

        # Reset gradients
        optimizer.zero_grad(set_to_none=True)

        # Forward pass
        logits = model(
            batch["user_ids"],
            batch["news_ids"],
            batch["category_ids"],
            batch["subcategory_ids"],
            batch["numerical_features"]
        )

        # Calculate loss
        loss = criterion(logits, batch["labels"])

        # Backpropagation
        loss.backward()

        # Update model weights
        optimizer.step()

        # Record loss
        batch_loss = loss.item()
        epoch_losses.append(batch_loss)

        running_loss += batch_loss
        running_batches += 1
        total_batches += 1

        del batch, logits, loss

        # Display progress
        if batch_number % LOG_EVERY == 0:
            average_loss = running_loss / running_batches
            elapsed = time.time() - start_time

            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Batch {batch_number} | "
                f"Average loss: {average_loss:.4f} | "
                f"Elapsed: {elapsed / 60:.1f} min"
            )

            running_loss = 0.0
            running_batches = 0

        # Save checkpoint periodically
        if batch_number % SAVE_EVERY == 0:

            checkpoint = {
                "epoch": epoch + 1,
                "batch": batch_number,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "losses": epoch_losses
            }

            torch.save(checkpoint, temp_checkpoint_path)
            os.replace(temp_checkpoint_path, checkpoint_path)

            print(
                f"Checkpoint saved after "
                f"{batch_number} batches."
            )

    del train_iterator
    gc.collect()

# Save the final checkpoint
final_checkpoint = {
    "epoch": EPOCHS,
    "batch": total_batches,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "losses": epoch_losses
}

torch.save(final_checkpoint, temp_checkpoint_path)
os.replace(temp_checkpoint_path, checkpoint_path)

elapsed_time = time.time() - start_time

print("\n" + "=" * 55)
print("TRAINING COMPLETED")
print("=" * 55)

print("Total batches:", total_batches)
print("Total candidates processed:", total_batches * BATCH_SIZE)
print("Average training loss:", round(np.mean(epoch_losses), 4))
print("Training time:", round(elapsed_time / 60, 2), "minutes")
print("Checkpoint saved:", checkpoint_path)

Starting full FIT training...
Training candidates: 12894210
Batch size: 2048
Checkpoint location: C:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\outputs\feature_mlp\feature_mlp_latest.pt
-------------------------------------------------------
Epoch 1/1 | Batch 250 | Average loss: 0.4897 | Elapsed: 0.7 min
Epoch 1/1 | Batch 500 | Average loss: 0.4907 | Elapsed: 1.4 min
Epoch 1/1 | Batch 750 | Average loss: 0.4723 | Elapsed: 2.1 min
Epoch 1/1 | Batch 1000 | Average loss: 0.4824 | Elapsed: 2.8 min
Checkpoint saved after 1000 batches.
Epoch 1/1 | Batch 1250 | Average loss: 0.4834 | Elapsed: 3.5 min
Epoch 1/1 | Batch 1500 | Average loss: 0.4820 | Elapsed: 4.2 min
Epoch 1/1 | Batch 1750 | Average loss: 0.4771 | Elapsed: 4.9 min
Epoch 1/1 | Batch 2000 | Average loss: 0.4731 | Elapsed: 5.7 min
Checkpoint saved after 2000 batches.
Epoch 1/1 | Batch 2250 | Average loss: 0.4679 | Elapsed: 6.5 min
Epoch 1/1 | Batch 2500 | Average loss: 0.4780

In [ ]:
print("FIT Parquet rows:", fit_parquet.metadata.num_rows)
print("Expected FIT rows:", 12_894_210)
print("Processed candidates:", total_batches * BATCH_SIZE)
print("Training batches:", total_batches)

FIT Parquet rows: 12894210
Expected FIT rows: 12894210
Processed candidates: 12896256
Training batches: 6297


In [ ]:
import inspect

print("evaluate:")
print(inspect.signature(evaluate))

print("\nevaluate_by_segment:")
print(inspect.signature(evaluate_by_segment))

print("\ntopk:")
print(inspect.signature(topk))

print("\ntopk_diversity:")
print(inspect.signature(topk_diversity))

evaluate:
(df, score_col, label_col='label', group_col='impression_id', per_impression=False, seed=42)

evaluate_by_segment:
(df, score_col, flag_col, label_col='label', group_col='impression_id', seed=42)

topk:
(df, rank_col, k=10, group_col='impression_id', ascending=True)

topk_diversity:
(top, category_col='category', emb_lookup=None, news_col='news_id', cold_col=None, group_col='impression_id', k=None)


In [ ]:
import inspect

print("===== evaluate() =====")
print(inspect.getsource(evaluate))

print("\n===== evaluate_by_segment() =====")
print(inspect.getsource(evaluate_by_segment))

print("\n===== topk() =====")
print(inspect.getsource(topk))

print("\n===== topk_diversity() =====")
print(inspect.getsource(topk_diversity))

===== evaluate() =====
def evaluate(df, score_col, label_col="label", group_col="impression_id", per_impression=False,
             seed=SEED):
    """Mean AUC, MRR, nDCG@5, nDCG@10 over impressions.

    Impressions with no click or no non-click are skipped (AUC undefined)
    and counted in 'n_skipped'.
    """
    res = _per_impression(df, score_col, label_col, group_col, seed)
    if per_impression:
        ok = res[res["valid"]]
        return ok[["auc", "mrr", "ndcg@5", "ndcg@10", "first_hit_rank", group_col]].reset_index(drop=True)
    return _summary(res)


===== evaluate_by_segment() =====
def evaluate_by_segment(df, score_col, flag_col, label_col="label", group_col="impression_id",
                        seed=SEED):
    """Relevance metrics overall and split into cold / warm / mixed impressions."""
    res = _per_impression(df, score_col, label_col, group_col, seed)
    res["segment"] = res[group_col].map(label_impression_segment(df, flag_col, label_col, group_col))
    rows

In [ ]:
import inspect
from utils import metrics

print("===== _per_impression() =====")
print(inspect.getsource(metrics._per_impression))

print("\n===== label_impression_segment() =====")
print(inspect.getsource(metrics.label_impression_segment))

print("\n===== topk() =====")
print(inspect.getsource(metrics.topk))

print("\n===== topk_diversity() =====")
print(inspect.getsource(metrics.topk_diversity))

===== _per_impression() =====
def _per_impression(df, score_col, label_col, group_col, seed=SEED):
    """One row per impression: auc, mrr, ndcg@5, ndcg@10, first_hit_rank (vectorised)."""
    iid = pd.factorize(df[group_col])[0]
    score = df[score_col].to_numpy(dtype=float)
    order, grp, first, rank = _rank_within_group(iid, score, seed)
    label_s = df[label_col].to_numpy(dtype=int)[order]
    score_s = score[order]

    n_cand = np.bincount(grp)
    n_pos = np.bincount(grp, weights=label_s)
    n_neg = n_cand - n_pos
    res = pd.DataFrame({group_col: df[group_col].to_numpy()[order][first]})

    with np.errstate(divide="ignore", invalid="ignore"):
        res["mrr"] = np.bincount(grp, weights=label_s / rank) / n_pos
        for k in KS:
            dcg = np.bincount(grp, weights=label_s * (rank <= k) / np.log2(rank + 1))
            ideal = np.r_[0.0, np.cumsum(1.0 / np.log2(np.arange(1, k + 1) + 1))]
            res[f"ndcg@{k}"] = dcg / ideal[np.minimum(n_pos, k).astype(int)]

In [ ]:
import inspect
from utils import metrics

print("===== _rank_within_group =====")
print(inspect.getsource(metrics._rank_within_group))

print("\n===== label_impression_segment =====")
print(inspect.getsource(metrics.label_impression_segment))

print("\n===== topk =====")
print(inspect.getsource(metrics.topk))

print("\n===== topk_diversity =====")
print(inspect.getsource(metrics.topk_diversity))

===== _rank_within_group =====
def _rank_within_group(iid, score, seed):
    """Sort rows by (impression, score desc, random tie-break) and rank inside each impression."""
    tie = np.random.default_rng(seed).random(len(score))
    order = np.lexsort((tie, -score, iid))
    iid_s = iid[order]
    new_grp = np.r_[True, iid_s[1:] != iid_s[:-1]]
    grp = np.cumsum(new_grp) - 1
    first = np.flatnonzero(new_grp)
    rank = np.arange(len(iid_s)) - first[grp] + 1
    return order, grp, first, rank


===== label_impression_segment =====
def label_impression_segment(df, flag_col, label_col="label", group_col="impression_id"):
    """Label each impression by its CLICKED articles' flag (e.g. is_cold).

    'cold'  = all clicked articles have flag 1
    'warm'  = all clicked articles have flag 0
    'mixed' = some of each
    Returns a Series indexed by impression_id.
    """
    clicked = df[df[label_col] == 1].groupby(group_col)[flag_col].agg(["min", "max"])
    seg = np.where(clicked["min"]

In [ ]:
import inspect

print("===== topk() =====")
print(inspect.getsource(metrics.topk))

print("\n===== topk_diversity() =====")
print(inspect.getsource(metrics.topk_diversity))

===== topk() =====
def topk(df, rank_col, k=10, group_col="impression_id", ascending=True):
    """Top-k rows per impression by rank_col (rank 1 = best when ascending).

    No rank column yet?  df = add_rank(df, "score")  then  topk(df, "rank").
    """
    return (df.sort_values([group_col, rank_col], ascending=[True, ascending])
              .groupby(group_col, sort=False).head(k))


===== topk_diversity() =====
def topk_diversity(top, category_col="category", emb_lookup=None, news_col="news_id",
                   cold_col=None, group_col="impression_id", k=None):
    """Diversity / exposure metrics for a top-k DataFrame (output of topk()).

    - unique_categories: mean number of distinct categories per list
    - intra_list_sim: mean pairwise cosine similarity (needs emb_lookup:
      dict news_id -> vector); lower = more diverse
    - cold_share: mean share of cold articles per list (needs cold_col)

    Only lists with exactly k items are used (k defaults to the longest list).

In [ ]:
import pyarrow.parquet as pq

tune_path = config.PROCESSED_DIR + "/tune_candidates.parquet"
tune_parquet = pq.ParquetFile(tune_path)

print("TUNE rows:", tune_parquet.metadata.num_rows)
print("TUNE row groups:", tune_parquet.num_row_groups)

sample = next(
    tune_parquet.iter_batches(
        batch_size=10000,
        columns=["impression_id"]
    )
).to_pandas()

print("\nFirst 20 impression IDs:")
print(sample["impression_id"].head(20).tolist())

print("\nAre impression IDs sorted in this sample?")
print(sample["impression_id"].is_monotonic_increasing)

TUNE rows: 17400106
TUNE row groups: 35

First 20 impression IDs:
['612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270', '612270']

Are impression IDs sorted in this sample?
False


In [ ]:
import numpy as np

completed_impressions = set()
current_impression = None
reappeared_groups = 0
total_groups = 0

for arrow_batch in tune_parquet.iter_batches(
    batch_size=100000,
    columns=["impression_id"]
):
    ids = arrow_batch.column("impression_id").to_numpy(
        zero_copy_only=False
    )

    change_positions = np.flatnonzero(
        np.r_[True, ids[1:] != ids[:-1]]
    )

    group_starts = ids[change_positions]

    for impression_id in group_starts:

        if current_impression is not None and impression_id != current_impression:

            completed_impressions.add(current_impression)

            if impression_id in completed_impressions:
                reappeared_groups += 1

        if impression_id != current_impression:
            total_groups += 1
            current_impression = impression_id

print("Total impression groups encountered:", total_groups)
print("Groups that reappeared after ending:", reappeared_groups)
print("Are impression groups contiguous:", reappeared_groups == 0)

del completed_impressions

Total impression groups encountered: 431517
Groups that reappeared after ending: 0
Are impression groups contiguous: True


In [ ]:
import torch
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path
import gc

# Load the trained checkpoint
checkpoint_path = (
    Path(PROJECT_ROOT)
    / "outputs"
    / "feature_mlp"
    / "feature_mlp_latest.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model checkpoint loaded successfully.")
print("Training epoch:", checkpoint["epoch"])
print("Training batches:", checkpoint["batch"])

# Open TUNE dataset
tune_path = Path(config.PROCESSED_DIR) / "tune_candidates.parquet"
tune_parquet = pq.ParquetFile(tune_path)

print("\nTUNE candidates:", tune_parquet.metadata.num_rows)
print("TUNE row groups:", tune_parquet.num_row_groups)

Model checkpoint loaded successfully.
Training epoch: 1
Training batches: 6297

TUNE candidates: 17400106
TUNE row groups: 35


In [ ]:
checkpoint_path = (
    Path(PROJECT_ROOT)
    / "outputs"
    / "feature_mlp"
    / "feature_mlp_latest.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Model loaded successfully.")
print("Epoch:", checkpoint["epoch"])
print("Training batches:", checkpoint["batch"])

tune_path = Path(config.PROCESSED_DIR) / "tune_candidates.parquet"
tune_parquet = pq.ParquetFile(tune_path)

print("TUNE candidates:", tune_parquet.metadata.num_rows)
print("TUNE impressions:", 431517)

Model loaded successfully.
Epoch: 1
Training batches: 6297
TUNE candidates: 17400106
TUNE impressions: 431517


In [ ]:

import time
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# Output location
prediction_dir = Path(PROJECT_ROOT) / "outputs" / "feature_mlp"
prediction_dir.mkdir(parents=True, exist_ok=True)

tune_prediction_path = prediction_dir / "tune_predictions.parquet"

# Columns required for feature transformation and evaluation
PREDICTION_COLUMNS = [
    "impression_id",
    "news_id",
    "user_idx",
    "news_idx",
    "prior_exposure_count",
    "cold_article",
    "category",
    "subcategory",
    "time",
    "label"
]

# Prepare streaming prediction
model.eval()

prediction_writer = None
processed_candidates = 0
start_time = time.time()

print("Starting TUNE prediction...")
print("Total candidates:", tune_parquet.metadata.num_rows)
print("Read batch size:", READ_BATCH_SIZE)
print("Output:", tune_prediction_path)

try:
    with torch.inference_mode():

        for batch_number, arrow_batch in enumerate(
            tune_parquet.iter_batches(
                batch_size=READ_BATCH_SIZE,
                columns=PREDICTION_COLUMNS
            ),
            start=1
        ):

            # Convert current batch into pandas
            batch_df = arrow_batch.to_pandas()

            # Apply the same feature transformations used during training
            tensors = transform_batch(batch_df)

            # Generate click probabilities
            logits = model(
                tensors["user_ids"],
                tensors["news_ids"],
                tensors["category_ids"],
                tensors["subcategory_ids"],
                tensors["numerical_features"]
            )

            probabilities = torch.sigmoid(logits).cpu().numpy()

            # Keep the original information needed for evaluation
            output_df = batch_df[
                [
                    "impression_id",
                    "news_id",
                    "label",
                    "category",
                    "cold_article"
                ]
            ].copy()

            output_df["score"] = probabilities

            # Write this batch directly to Parquet
            output_table = pa.Table.from_pandas(
                output_df,
                preserve_index=False
            )

            if prediction_writer is None:
                prediction_writer = pq.ParquetWriter(
                    tune_prediction_path,
                    output_table.schema,
                    compression="zstd"
                )

            prediction_writer.write_table(output_table)

            processed_candidates += len(batch_df)

            if batch_number % 100 == 0:
                elapsed = time.time() - start_time
                progress = (
                    processed_candidates
                    / tune_parquet.metadata.num_rows
                    * 100
                )

                print(
                    f"Batch {batch_number} | "
                    f"Processed: {processed_candidates:,} | "
                    f"Progress: {progress:.1f}% | "
                    f"Elapsed: {elapsed / 60:.1f} min"
                )

            del batch_df, tensors, logits
            del probabilities, output_df, output_table, arrow_batch

finally:
    if prediction_writer is not None:
        prediction_writer.close()

    gc.collect()

elapsed = time.time() - start_time

print("\nTUNE PREDICTION COMPLETED")
print("Candidates processed:", f"{processed_candidates:,}")
print("Expected candidates:", f"{tune_parquet.metadata.num_rows:,}")
print("Elapsed time:", f"{elapsed / 60:.2f} minutes")
print("Saved to:", tune_prediction_path)


Starting TUNE prediction...
Total candidates: 17400106
Read batch size: 16384
Output: C:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\outputs\feature_mlp\tune_predictions.parquet
Batch 100 | Processed: 1,638,400 | Progress: 9.4% | Elapsed: 0.1 min
Batch 200 | Processed: 3,276,800 | Progress: 18.8% | Elapsed: 0.3 min
Batch 300 | Processed: 4,915,200 | Progress: 28.2% | Elapsed: 0.4 min
Batch 400 | Processed: 6,553,600 | Progress: 37.7% | Elapsed: 0.6 min
Batch 500 | Processed: 8,192,000 | Progress: 47.1% | Elapsed: 0.7 min
Batch 600 | Processed: 9,830,400 | Progress: 56.5% | Elapsed: 0.9 min
Batch 700 | Processed: 11,468,800 | Progress: 65.9% | Elapsed: 1.0 min
Batch 800 | Processed: 13,107,200 | Progress: 75.3% | Elapsed: 1.2 min
Batch 900 | Processed: 14,745,600 | Progress: 84.7% | Elapsed: 1.3 min
Batch 1000 | Processed: 16,384,000 | Progress: 94.2% | Elapsed: 1.4 min

TUNE PREDICTION COMPLETED
Candidates processed: 17,400,106
Ex

In [ ]:
#Before calculating recommendation metrics, we verify that the prediction file contains exactly the expected number of candidates and that the MLP scores are valid probabilities.
import pyarrow.parquet as pq
import numpy as np

prediction_parquet = pq.ParquetFile(tune_prediction_path)

print("Prediction rows:", prediction_parquet.metadata.num_rows)
print("Expected rows:", tune_parquet.metadata.num_rows)
print("Row groups:", prediction_parquet.num_row_groups)

sample = next(
    prediction_parquet.iter_batches(
        batch_size=10000,
        columns=[
            "impression_id",
            "news_id",
            "label",
            "category",
            "cold_article",
            "score"
        ]
    )
).to_pandas()

print("\nSample:")
print(sample.head())

print("\nScore statistics:")
print(sample["score"].describe())

print("\nScore range:")
print("Minimum:", sample["score"].min())
print("Maximum:", sample["score"].max())

print("\nLabel distribution:")
print(sample["label"].value_counts().sort_index())

Prediction rows: 17400106
Expected rows: 17400106
Row groups: 1063

Sample:
  impression_id  news_id  label       category  cold_article     score
0        612270  N123683      0      lifestyle             0  0.320105
1        612270  N108269      0           news             0  0.067060
2        612270  N120332      0           news             0  0.226614
3        612270   N18405      0           news             0  0.187917
4        612270   N51569      0  entertainment             0  0.091615

Score statistics:
count    10000.000000
mean         0.177436
std          0.086438
min          0.013000
25%          0.112883
50%          0.168071
75%          0.236718
max          0.623732
Name: score, dtype: float64

Score range:
Minimum: 0.013000223
Maximum: 0.6237319

Label distribution:
label
0    9586
1     414
Name: count, dtype: int64


In [ ]:

# Efficient TUNE evaluation
# Processes multiple complete impressions per evaluation call.

import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from pathlib import Path

# Locate the existing TUNE predictions
PREDICTIONS_PATH = (
    PROJECT_ROOT / "outputs" / "feature_mlp" / "tune_predictions.parquet"
)

if not PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        f"TUNE predictions not found: {PREDICTIONS_PATH}"
    )

print("Reading:", PREDICTIONS_PATH)

# Read only the columns needed for the main ranking metrics
parquet_file = pq.ParquetFile(PREDICTIONS_PATH)

metric_parts = []
pending = None

total_impressions = 0
batch_number = 0
BATCH_SIZE = 100_000

for arrow_batch in parquet_file.iter_batches(
    batch_size=BATCH_SIZE,
    columns=["impression_id", "label", "score"]
):
    batch_number += 1
    batch_df = arrow_batch.to_pandas()

    # Retain the unfinished impression from the previous batch
    if pending is not None:
        combined = pd.concat(
            [pending, batch_df],
            ignore_index=True
        )
    else:
        combined = batch_df

    # Keep the final impression for the next batch.
    # This prevents an impression from being split across evaluations.
    last_impression = combined["impression_id"].iloc[-1]

    last_group_start = np.flatnonzero(
        combined["impression_id"].to_numpy() == last_impression
    )[0]

    complete = combined.iloc[:last_group_start].copy()
    pending = combined.iloc[last_group_start:].copy()

    # Evaluate all complete impressions together
    if not complete.empty:
        total_impressions += complete["impression_id"].nunique()

        batch_metrics = metrics_module.evaluate(
            complete,
            score_col="score",
            label_col="label",
            group_col="impression_id",
            per_impression=True
        )

        metric_parts.append(batch_metrics)

    if batch_number % 25 == 0:
        print(
            f"Processed batches: {batch_number} | "
            f"Impressions completed: {total_impressions:,}"
        )

    del arrow_batch, batch_df, combined, complete

# Process the final pending impression
if pending is not None and not pending.empty:
    total_impressions += pending["impression_id"].nunique()

    final_metrics = metrics_module.evaluate(
        pending,
        score_col="score",
        label_col="label",
        group_col="impression_id",
        per_impression=True
    )

    metric_parts.append(final_metrics)

    del pending

# Combine per-impression results
if not metric_parts:
    raise ValueError("No evaluation results were produced.")

tune_metrics = pd.concat(
    metric_parts,
    ignore_index=True
)

# Calculate the same mean metrics as the team's _summary function
summary = (
    tune_metrics[["auc", "mrr", "ndcg@5", "ndcg@10"]]
    .mean()
    .round(4)
    .to_dict()
)

summary["n_impressions"] = len(tune_metrics)
summary["n_skipped"] = total_impressions - len(tune_metrics)

# Display results
print("\nTUNE EVALUATION RESULTS")
print("=======================")

for metric, value in summary.items():
    print(f"{metric}: {value}")

# Save per-impression metrics for later analysis
OUTPUT_PATH = (
    PROJECT_ROOT / "outputs" / "feature_mlp"
    / "tune_per_impression_metrics.parquet"
)

tune_metrics.to_parquet(OUTPUT_PATH, index=False)

print("\nPer-impression metrics saved to:")
print(OUTPUT_PATH)

Reading: C:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\outputs\feature_mlp\tune_predictions.parquet
Processed batches: 25 | Impressions completed: 55,958
Processed batches: 50 | Impressions completed: 116,282
Processed batches: 75 | Impressions completed: 178,351
Processed batches: 100 | Impressions completed: 238,870
Processed batches: 125 | Impressions completed: 301,115
Processed batches: 150 | Impressions completed: 367,983
Processed batches: 175 | Impressions completed: 431,516

TUNE EVALUATION RESULTS
auc: 0.5672
mrr: 0.2333
ndcg@5: 0.2398
ndcg@10: 0.3026
n_impressions: 431517
n_skipped: 0

Per-impression metrics saved to:
C:\Users\eishw\OneDrive - National University of Singapore\BT4222\Project\BT4222-Project-Group-4--MIND\outputs\feature_mlp\tune_per_impression_metrics.parquet


In [ ]:
#checking
import inspect
import utils.metrics as metrics_module

print("EVALUATE FUNCTION")
print(inspect.signature(metrics_module.evaluate))
print(inspect.getsource(metrics_module.evaluate))

print("\nPER-IMPRESSION FUNCTION")
print(inspect.getsource(metrics_module._per_impression))

EVALUATE FUNCTION
(df, score_col, label_col='label', group_col='impression_id', per_impression=False, seed=42)
def evaluate(df, score_col, label_col="label", group_col="impression_id", per_impression=False,
             seed=SEED):
    """Mean AUC, MRR, nDCG@5, nDCG@10 over impressions.

    Impressions with no click or no non-click are skipped (AUC undefined)
    and counted in 'n_skipped'.
    """
    res = _per_impression(df, score_col, label_col, group_col, seed)
    if per_impression:
        ok = res[res["valid"]]
        return ok[["auc", "mrr", "ndcg@5", "ndcg@10", "first_hit_rank", group_col]].reset_index(drop=True)
    return _summary(res)


PER-IMPRESSION FUNCTION
def _per_impression(df, score_col, label_col, group_col, seed=SEED):
    """One row per impression: auc, mrr, ndcg@5, ndcg@10, first_hit_rank (vectorised)."""
    iid = pd.factorize(df[group_col])[0]
    score = df[score_col].to_numpy(dtype=float)
    order, grp, first, rank = _rank_within_group(iid, score, seed)
 

In [ ]:
print(inspect.getsource(metrics_module._summary))

def _summary(res):
    """Mean metrics over the valid impressions of `res` (output of _per_impression)."""
    ok = res[res["valid"]]
    out = ok[["auc", "mrr", "ndcg@5", "ndcg@10"]].mean().round(4).to_dict()
    out["n_impressions"] = int(len(ok))
    out["n_skipped"] = int(len(res) - len(ok))
    return out



In [ ]:
import inspect

print("EVALUATE BY SEGMENT")
print(inspect.signature(metrics_module.evaluate_by_segment))
print(inspect.getsource(metrics_module.evaluate_by_segment))

print("\nTOP-K FUNCTION")
print(inspect.signature(metrics_module.topk))
print(inspect.getsource(metrics_module.topk))

print("\nTOP-K DIVERSITY")
print(inspect.signature(metrics_module.topk_diversity))
print(inspect.getsource(metrics_module.topk_diversity))

EVALUATE BY SEGMENT
(df, score_col, flag_col, label_col='label', group_col='impression_id', seed=42)
def evaluate_by_segment(df, score_col, flag_col, label_col="label", group_col="impression_id",
                        seed=SEED):
    """Relevance metrics overall and split into cold / warm / mixed impressions."""
    res = _per_impression(df, score_col, label_col, group_col, seed)
    res["segment"] = res[group_col].map(label_impression_segment(df, flag_col, label_col, group_col))
    rows = {"all": _summary(res)}
    for s in ["cold", "warm", "mixed"]:
        sub = res[res["segment"] == s]
        if len(sub):
            rows[s] = _summary(sub)
    out = pd.DataFrame(rows).T
    out[["n_impressions", "n_skipped"]] = out[["n_impressions", "n_skipped"]].astype(int)
    return out


TOP-K FUNCTION
(df, rank_col, k=10, group_col='impression_id', ascending=True)
def topk(df, rank_col, k=10, group_col="impression_id", ascending=True):
    """Top-k rows per impression by rank_col (rank 1 

In [ ]:
print("IMPRESSION SEGMENT LABELING")
print(inspect.getsource(metrics_module.label_impression_segment))

print("\nTOP-K DIVERSITY")
print(inspect.getsource(metrics_module.topk_diversity))

print("\nRANKING HELPER")
print(inspect.getsource(metrics_module.add_rank))

IMPRESSION SEGMENT LABELING
def label_impression_segment(df, flag_col, label_col="label", group_col="impression_id"):
    """Label each impression by its CLICKED articles' flag (e.g. is_cold).

    'cold'  = all clicked articles have flag 1
    'warm'  = all clicked articles have flag 0
    'mixed' = some of each
    Returns a Series indexed by impression_id.
    """
    clicked = df[df[label_col] == 1].groupby(group_col)[flag_col].agg(["min", "max"])
    seg = np.where(clicked["min"] == 1, "cold", np.where(clicked["max"] == 0, "warm", "mixed"))
    return pd.Series(seg, index=clicked.index, name="segment")


TOP-K DIVERSITY
def topk_diversity(top, category_col="category", emb_lookup=None, news_col="news_id",
                   cold_col=None, group_col="impression_id", k=None):
    """Diversity / exposure metrics for a top-k DataFrame (output of topk()).

    - unique_categories: mean number of distinct categories per list
    - intra_list_sim: mean pairwise cosine similarity (needs em

In [ ]:

# Cold/Warm Performance + Top-10 Diversity Evaluation

import pyarrow.parquet as pq
import pandas as pd
import numpy as np

PREDICTIONS_PATH = (
    PROJECT_ROOT / "outputs" / "feature_mlp"
    / "tune_predictions.parquet"
)

parquet_file = pq.ParquetFile(PREDICTIONS_PATH)

BATCH_SIZE = 100_000
METRIC_COLUMNS = ["auc", "mrr", "ndcg@5", "ndcg@10"]

# Accumulate weighted segment metrics
segment_totals = {}

# Accumulate top-10 diversity metrics
diversity_totals = {
    "unique_categories_sum": 0.0,
    "cold_share_sum": 0.0,
    "n_lists": 0,
    "n_short_lists_excluded": 0
}

pending = None
batch_number = 0
total_rows = 0

def process_evaluation_batch(df):

    global total_rows

    total_rows += len(df)

    # ----------------------------------
    # Cold / warm / mixed performance
    # ----------------------------------

    segment_results = metrics_module.evaluate_by_segment(
        df,
        score_col="score",
        flag_col="cold_article",
        label_col="label",
        group_col="impression_id"
    )

    for segment, row in segment_results.iterrows():

        if segment not in segment_totals:
            segment_totals[segment] = {
                **{metric: 0.0 for metric in METRIC_COLUMNS},
                "n_impressions": 0,
                "n_skipped": 0
            }

        valid_count = int(row["n_impressions"])

        for metric in METRIC_COLUMNS:
            if valid_count > 0:
                segment_totals[segment][metric] += (
                    float(row[metric]) * valid_count
                )

        segment_totals[segment]["n_impressions"] += valid_count
        segment_totals[segment]["n_skipped"] += int(row["n_skipped"])

    # ----------------------------------
    # Rank candidates and select top 10
    # ----------------------------------

    top10 = (
        df.sort_values(
            ["impression_id", "score"],
            ascending=[True, False]
        )
        .groupby("impression_id", sort=False)
        .head(10)
    )

    diversity_result = metrics_module.topk_diversity(
        top10,
        category_col="category",
        cold_col="cold_article",
        group_col="impression_id",
        k=10
    )

    n_lists = int(diversity_result["n_lists"])

    if n_lists > 0:
        diversity_totals["unique_categories_sum"] += (
            float(diversity_result["unique_categories"]) * n_lists
        )

        diversity_totals["cold_share_sum"] += (
            float(diversity_result["cold_share"]) * n_lists
        )

        diversity_totals["n_lists"] += n_lists

    diversity_totals["n_short_lists_excluded"] += int(
        diversity_result["n_short_lists_excluded"]
    )


# ----------------------------------
# Stream TUNE predictions
# ----------------------------------

for arrow_batch in parquet_file.iter_batches(
    batch_size=BATCH_SIZE,
    columns=[
        "impression_id",
        "news_id",
        "label",
        "category",
        "cold_article",
        "score"
    ]
):

    batch_number += 1
    batch_df = arrow_batch.to_pandas()

    if pending is not None:
        combined = pd.concat(
            [pending, batch_df],
            ignore_index=True
        )
    else:
        combined = batch_df

    # Retain the final impression to avoid splitting groups
    last_impression = combined["impression_id"].iloc[-1]

    last_group_start = np.flatnonzero(
        combined["impression_id"].to_numpy() == last_impression
    )[0]

    complete = combined.iloc[:last_group_start].copy()
    pending = combined.iloc[last_group_start:].copy()

    if not complete.empty:
        process_evaluation_batch(complete)

    if batch_number % 25 == 0:
        print(
            f"Batches processed: {batch_number} | "
            f"Rows processed: {total_rows:,}"
        )

    del arrow_batch, batch_df, combined, complete

# Process final remaining impression
if pending is not None and not pending.empty:
    process_evaluation_batch(pending)

del pending

# ----------------------------------
# Display cold/warm results
# ----------------------------------

segment_rows = []

for segment, values in segment_totals.items():

    n = values["n_impressions"]

    row = {"segment": segment}

    for metric in METRIC_COLUMNS:
        row[metric] = (
            round(values[metric] / n, 4)
            if n > 0 else np.nan
        )

    row["n_impressions"] = n
    row["n_skipped"] = values["n_skipped"]

    segment_rows.append(row)

segment_summary = (
    pd.DataFrame(segment_rows)
    .set_index("segment")
)

print("\nCOLD / WARM PERFORMANCE")
print("=======================")
display(segment_summary)

# ----------------------------------
# Display top-10 diversity results
# ----------------------------------

n_lists = diversity_totals["n_lists"]

if n_lists > 0:

    diversity_summary = pd.DataFrame([{
        "unique_categories": round(
            diversity_totals["unique_categories_sum"] / n_lists, 4
        ),
        "cold_share": round(
            diversity_totals["cold_share_sum"] / n_lists, 4
        ),
        "n_lists": n_lists,
        "n_short_lists_excluded":
            diversity_totals["n_short_lists_excluded"]
    }])

    print("\nTOP-10 DIVERSITY AND EXPOSURE")
    print("============================")
    display(diversity_summary)

else:
    raise ValueError("No valid top-10 recommendation lists found.")

# ----------------------------------
# Save results
# ----------------------------------

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "feature_mlp"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

segment_summary.to_csv(
    OUTPUT_DIR / "tune_segment_metrics.csv"
)

diversity_summary.to_csv(
    OUTPUT_DIR / "tune_diversity_metrics.csv",
    index=False
)

print("\nEvaluation completed.")
print("Rows processed:", f"{total_rows:,}")
print("Segment results saved.")
print("Diversity results saved.")


Batches processed: 25 | Rows processed: 2,499,992
Batches processed: 50 | Rows processed: 4,999,965
Batches processed: 75 | Rows processed: 7,499,963
Batches processed: 100 | Rows processed: 9,999,982
Batches processed: 125 | Rows processed: 12,499,978
Batches processed: 150 | Rows processed: 14,999,968
Batches processed: 175 | Rows processed: 17,400,093

COLD / WARM PERFORMANCE


,auc,mrr,ndcg@5,ndcg@10,n_impressions,n_skipped
segment,,,,,,
all,0.5672,0.2333,0.2398,0.3026,431517,0
cold,0.7346,0.4444,0.4793,0.5278,98756,0
warm,0.5053,0.1740,0.1670,0.2343,289215,0
mixed,0.5988,0.1485,0.1800,0.2450,43546,0



TOP-10 DIVERSITY AND EXPOSURE


,unique_categories,cold_share,n_lists,n_short_lists_excluded
0,6.2707,0.4156,348203,83314



Evaluation completed.
Rows processed: 17,400,106
Segment results saved.
Diversity results saved.


In [ ]:
print("COLD / WARM PERFORMANCE")
print(segment_summary.to_string())

print("\nTOP-10 DIVERSITY AND EXPOSURE")
print(diversity_summary.to_string(index=False))

COLD / WARM PERFORMANCE
            auc     mrr  ndcg@5  ndcg@10  n_impressions  n_skipped
segment                                                           
all      0.5672  0.2333  0.2398   0.3026         431517          0
cold     0.7346  0.4444  0.4793   0.5278          98756          0
warm     0.5053  0.1740  0.1670   0.2343         289215          0
mixed    0.5988  0.1485  0.1800   0.2450          43546          0

TOP-10 DIVERSITY AND EXPOSURE
 unique_categories  cold_share  n_lists  n_short_lists_excluded
            6.2707      0.4156   348203                   83314


In [ ]:

import time
import gc
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# ----------------------------------
# Locate EVAL candidate data
# ----------------------------------

# Reuse the same processed-data directory as TUNE
eval_candidates_path = (
    Path(tune_parquet.path).parent / "eval_candidates.parquet"
)

eval_parquet = pq.ParquetFile(eval_candidates_path)

# ----------------------------------
# Output location
# ----------------------------------

prediction_dir = Path(PROJECT_ROOT) / "outputs" / "feature_mlp"
prediction_dir.mkdir(parents=True, exist_ok=True)

eval_prediction_path = prediction_dir / "eval_predictions.parquet"

# ----------------------------------
# Columns required for prediction
# ----------------------------------

PREDICTION_COLUMNS = [
    "impression_id",
    "news_id",
    "user_idx",
    "news_idx",
    "prior_exposure_count",
    "cold_article",
    "category",
    "subcategory",
    "time",
    "label"
]

# ----------------------------------
# Prepare streaming prediction
# ----------------------------------

model.eval()

prediction_writer = None
processed_candidates = 0
start_time = time.time()

print("Starting EVAL prediction...")
print("Input:", eval_candidates_path)
print("Total candidates:", eval_parquet.metadata.num_rows)
print("Read batch size:", READ_BATCH_SIZE)
print("Output:", eval_prediction_path)

try:
    with torch.inference_mode():

        for batch_number, arrow_batch in enumerate(
            eval_parquet.iter_batches(
                batch_size=READ_BATCH_SIZE,
                columns=PREDICTION_COLUMNS
            ),
            start=1
        ):

            # Convert current batch into pandas
            batch_df = arrow_batch.to_pandas()

            # Apply the same feature transformations used during training
            tensors = transform_batch(batch_df)

            # Generate click probabilities
            logits = model(
                tensors["user_ids"],
                tensors["news_ids"],
                tensors["category_ids"],
                tensors["subcategory_ids"],
                tensors["numerical_features"]
            )

            probabilities = torch.sigmoid(logits).cpu().numpy()

            # Keep original information needed for evaluation
            output_df = batch_df[
                [
                    "impression_id",
                    "news_id",
                    "label",
                    "category",
                    "cold_article"
                ]
            ].copy()

            output_df["score"] = probabilities

            # Write batch directly to Parquet
            output_table = pa.Table.from_pandas(
                output_df,
                preserve_index=False
            )

            if prediction_writer is None:
                prediction_writer = pq.ParquetWriter(
                    eval_prediction_path,
                    output_table.schema,
                    compression="zstd"
                )

            prediction_writer.write_table(output_table)

            processed_candidates += len(batch_df)

            if batch_number % 100 == 0:
                elapsed = time.time() - start_time

                progress = (
                    processed_candidates
                    / eval_parquet.metadata.num_rows
                    * 100
                )

                print(
                    f"Batch {batch_number} | "
                    f"Processed: {processed_candidates:,} | "
                    f"Progress: {progress:.1f}% | "
                    f"Elapsed: {elapsed / 60:.1f} min"
                )

            del batch_df, tensors, logits
            del probabilities, output_df, output_table, arrow_batch

finally:
    if prediction_writer is not None:
        prediction_writer.close()

    gc.collect()

elapsed = time.time() - start_time

print("\nEVAL PREDICTION COMPLETED")
print("Candidates processed:", f"{processed_candidates:,}")
print("Expected candidates:", f"{eval_parquet.metadata.num_rows:,}")
print("Elapsed time:", f"{elapsed / 60:.2f} minutes")
print("Saved to:", eval_prediction_path)


AttributeError: 'ParquetFile' object has no attribute 'path'